# 📗 부록 02 심화: Ruff 코드 검사와 AI 코드 리뷰를 워크플로에 더하기

부록 02 기초에서는 pytest 테스트를 돌리는 워크플로를 저장소에 올리고, 요구사항을 보고 `ci.yml`을 설계했습니다. 이 부록은 같은 연습 프로젝트(`pytest_ci_practice/`)에 테스트로는 잡기 어려운 문제를 찾는 검사 두 가지를 더합니다.

- **Ruff:** 코드를 실행하지 않고 읽어서 실수와 모양 차이를 찾고 고칩니다. API 키 없이 따라 할 수 있습니다.
- **AI 코드 리뷰:** pull request마다 LLM이 바뀐 코드를 읽고 의견을 댓글로 남깁니다. `OPENAI_API_KEY`가 필요하고, pull request마다 호출 비용이 듭니다.

부록 02 기초 2절에서 만든 저장소와 기초 4-2의 비밀 변수 등록 방법을 그대로 씁니다. 이 부록의 명령은 모두 터미널에서 `pytest_ci_practice` 폴더(저장소 최상위)로 이동해 실행합니다.

**학습 목표**

- 테스트, 린트, 포맷, AI 리뷰가 각각 무엇을 확인하는지 구분합니다.
- Ruff 검사 결과에서 파일, 위치, 규칙 코드, 자동 고침 표시를 읽고, 규칙마다 고치는 방법을 고릅니다.
- 확인만 하는 Ruff 명령과 파일을 실제로 고치는 Ruff 명령을 구분해 씁니다.
- Ruff 검사를 워크플로 단계로 넣어, 규칙을 어긴 변경이 검사에서 실패하게 합니다.
- pull request마다 LLM이 바뀐 코드를 리뷰해 댓글로 남기는 작업을 워크플로에 더합니다.

<img src="../images/ap_20_four_checks.png" width="1050" alt="변경 요청을 확인하는 네 가지 방법. 테스트(pytest)는 실행해서 결과를 확인하고, 린트(ruff check)는 실행하지 않고 읽어서 실수를 찾고, 포맷(ruff format --check)은 동작은 그대로 두고 모양만 정리하며, 이 셋은 통과와 실패를 판정합니다. AI 리뷰(LLM)는 바뀐 코드를 읽고 참고 의견을 남깁니다.">

## 1. 린트와 포맷

### 1-1. 테스트가 못 잡는 문제

pytest는 코드를 **실행해서** 결과가 기대한 값과 같은지 봅니다. 그래서 테스트가 실행하지 않은 줄은 확인하지 못하고, 결과에 영향이 없는 차이는 문제로 보지 않습니다. 테스트를 모두 통과해도 다음과 같은 것이 남을 수 있습니다.

- **쓰지 않는 import와 변수:** 읽는 사람이 어디에 쓰는지 찾아 헤맵니다.
- **테스트가 실행하지 않은 줄의 오타:** 그 줄이 실제로 실행될 때 처음 오류가 납니다.
- **사람마다 다른 공백, 따옴표, 빈 줄:** pull request의 바뀐 줄에 내용과 상관없는 변경이 섞입니다.

이런 문제는 코드를 실행하지 않고 읽기만 해도 찾을 수 있습니다. 글쓰기에 비유하면 읽어서 실수를 찾는 **린트**는 맞춤법 검사이고, 모양을 맞추는 **포맷**은 글꼴과 줄 간격을 통일하는 서식 맞추기입니다. 규칙으로 적기 어려운 문제는 4절의 AI 리뷰가 의견으로 짚습니다.

| 검사 | 확인하는 방법 | 잡는 것 | 이 부록의 도구 |
|---|---|---|---|
| 테스트 | 코드를 실행해 결과를 기대값과 비교 | 결과가 틀린 코드 | `pytest` |
| 린트 | 실행하지 않고 코드를 읽어 규칙과 대조 | 안 쓰는 import·변수, 정의되지 않은 이름, 버그가 나기 쉬운 쓰임 | `ruff check` |
| 포맷 | 정해진 모양 규칙대로 다시 써 보고 지금 모양과 비교 | 공백, 빈 줄, 따옴표, 줄 나누기 차이 | `ruff format` |
| AI 리뷰 | LLM이 바뀐 코드를 읽고 의견을 씀 | 이름, 설계, 빠진 처리처럼 규칙으로 적기 어려운 문제 | `tools/ai_reviewer.py` |

### 1-2. 린트

**린트**(lint)는 코드를 실행하지 않고 읽어서 실수일 가능성이 높은 곳을 찾는 일이고, 이 일을 하는 도구를 **린터**(linter)라고 합니다. 워드프로세서가 문서를 읽으며 틀린 맞춤법에 빨간 줄을 긋듯, 린터는 코드를 읽으며 문제 있는 줄을 짚습니다. 이름은 1978년 C 언어 코드를 검사하던 프로그램 `lint`에서 왔고, 옷의 보풀(lint)을 떼어 내듯 작은 결함을 걸러 낸다는 뜻입니다.

코드를 실행하지 않으므로 테스트가 지나가지 않는 줄까지 모두 보고, API 키나 데이터 없이 몇 초 만에 끝납니다.

**예시:** 이름을 받아 인사말을 돌려주는 함수입니다. 안 좋은 예의 오른쪽 주석이 실수입니다.

안 좋은 예

```python
import os  # 불러오고 쓰지 않음


def greet(name):
    message = "반가워요"  # 값을 넣고 쓰지 않음
    if name == None:  # None 을 == 로 비교
        return "이름을 알려 주세요"
    return f"{nmae}님, 안녕하세요"  # name 을 nmae 로 잘못 씀
```

고친 예

```python
def greet(name):
    if name is None:
        return "이름을 알려 주세요"
    return f"{name}님, 안녕하세요"
```

- **쓰지 않는 import와 변수**(`import os`, `message`): 읽는 사람이 어디에 쓰는지 찾아 헤매고, 쓰려다 잊은 것인지 지우지 않은 찌꺼기인지 알 수 없습니다. 지웁니다.
- **`None`을 `==`로 비교**: 대부분은 같게 동작하지만 `==`의 뜻을 바꿔 둔 객체에서는 결과가 달라집니다. 파이썬 스타일 안내서 PEP 8은 `is None`을 권합니다.
- **이름 오타**(`nmae`): 이 코드가 실행되는 순간 `NameError`가 납니다. `name`으로 바로잡습니다.

이 실수들은 실행해 봐도 잘 드러나지 않습니다. `greet(None)`은 `if` 안에서 바로 `"이름을 알려 주세요"`를 돌려주므로 오타가 있는 마지막 `return`까지 가지 않고, 이름을 넣은 `greet("민수")`를 실행해야 비로소 `NameError: name 'nmae' is not defined`가 납니다. 나머지 세 실수는 어느 경우를 실행해도 오류 없이 지나가 코드에 계속 남습니다. 린터는 실행하지 않고 읽기만 하므로 네 가지를 한 번에 찾습니다. 각 실수가 어떤 규칙에 걸리는지는 1-5에서 봅니다.

린터가 짚은 곳이 모두 버그는 아닙니다. 린터의 말은 "규칙에 어긋난다"는 신호이므로, 고칠지 그 규칙만 예외로 둘지(2-5)는 사람이 정합니다.

### 1-3. 포맷

**포맷**(format)은 코드가 하는 일은 그대로 두고 공백, 빈 줄, 따옴표, 줄 나누기 같은 모양만 정해진 규칙대로 바꾸는 일이고, 이 일을 하는 도구를 **포맷터**(formatter)라고 합니다. 문서 전체의 글꼴과 줄 간격을 한 번에 맞추는 서식 기능과 같습니다. 규칙은 PEP 8의 권장(쉼표 뒤 공백 하나, 함수와 함수 사이 빈 줄 두 개 등)을 따릅니다.

**안 좋은 예:** 합계 금액을 계산하는 함수 두 개입니다. 쉼표 뒤와 `=`, `*`, `+` 양옆에 공백이 없고, 두 함수 사이에 빈 줄이 없고, 작은따옴표를 썼습니다. 실수는 아니어서 린트 검사(`ruff check`)는 통과합니다.

`price.py`

```python
def total_price(prices,tax_rate=0.1):
    subtotal=sum(prices)
    return round(subtotal*(1+tax_rate),2)
def price_label(price):
    return 'Total: '+str(price)
```

**고친 예:** `ruff format`으로 포맷한 결과입니다. 공백을 한 칸씩 넣고, 함수 사이에 빈 줄 두 개를 넣고, 따옴표를 큰따옴표로 바꿨습니다. 글자 배치만 달라졌을 뿐 두 파일은 같은 코드라 실행 결과도 같습니다.

```python
def total_price(prices, tax_rate=0.1):
    subtotal = sum(prices)
    return round(subtotal * (1 + tax_rate), 2)


def price_label(price):
    return "Total: " + str(price)
```

**모양을 왜 맞출까요?** 안 좋은 예의 파일을 연 사람이 세율 하나만 바꿨습니다(`0.1`을 `0.08`로). 그런데 그 사람의 편집기가 저장하면서 포맷까지 바꿨다면, pull request의 바뀐 줄은 이렇게 보입니다.

```diff
--- a/price.py
+++ b/price.py
@@ -1,5 +1,7 @@
-def total_price(prices,tax_rate=0.1):
-    subtotal=sum(prices)
-    return round(subtotal*(1+tax_rate),2)
+def total_price(prices, tax_rate=0.08):
+    subtotal = sum(prices)
+    return round(subtotal * (1 + tax_rate), 2)
+
+
 def price_label(price):
-    return 'Total: '+str(price)
+    return "Total: " + str(price)
```

줄 열 개가 바뀐 것으로 나오는데, 실제로 동작이 바뀐 곳은 `def total_price` 줄 안의 `0.08` 하나뿐입니다. 그 하나마저 공백 변경과 같은 줄에 섞여 있어 리뷰하는 사람이 놓치기 쉽습니다. 저장소의 코드가 처음부터 포맷되어 있었다면 같은 변경은 한 줄로만 보입니다.

```diff
--- a/price.py
+++ b/price.py
@@ -1,4 +1,4 @@
-def total_price(prices, tax_rate=0.1):
+def total_price(prices, tax_rate=0.08):
     subtotal = sum(prices)
     return round(subtotal * (1 + tax_rate), 2)
```

모두가 같은 포맷터를 쓰면 코드 모양이 한 가지로 정해집니다. 리뷰에서 모양을 두고 의견을 나눌 일이 없어지고, pull request의 바뀐 줄에는 내용이 바뀐 줄만 남습니다.

### 1-4. Ruff

**Ruff**는 린터와 포맷터를 한 도구에 담은 파이썬 도구입니다. 맞춤법 검사와 서식 맞추기를 한 프로그램이 하는 셈입니다. uv를 만든 Astral이 Rust로 만들어 빠릅니다. 린트는 `ruff check`, 포맷은 `ruff format` 명령으로 합니다.

연습 프로젝트의 `pyproject.toml`에 Ruff 설정이 들어 있습니다(`[Ruff 추가]` 주석이 붙은 곳). 설정은 이 파일 한 곳에 적고, 고치는 명령은 두 개입니다. 명령마다 하는 일은 2-2에서 봅니다.

`pyproject.toml`

```toml
[dependency-groups]
dev = [
    "ruff==0.17.1",
]

[tool.ruff]
line-length = 120          # 한 줄 최대 길이

[tool.ruff.lint]
select = ["E", "F", "I"]   # E: 코드 스타일, F: 문법 오류·안 쓰는 import, I: import 순서
```

```bash
uv run ruff check --fix .  # 규칙 검사와 import 순서 정리(파일을 고침)
uv run ruff format .       # 포맷 정리(파일을 고침)
```

`dev` 묶음은 개발할 때만 쓰는 도구 목록이고, `uv sync`가 기본으로 함께 설치합니다.

### 1-5. 규칙 묶음과 규칙 코드

Ruff의 린트 규칙마다 `F401` 같은 **규칙 코드**가 붙습니다. 앞 글자는 규칙 묶음, 뒤 숫자는 묶음 안의 번호입니다.

| 묶음 글자 | 묶음 이름 | 잡는 것 | 연습 프로젝트의 `select` |
|---|---|---|---|
| `F` | Pyflakes | 쓰지 않는 import·변수, 정의되지 않은 이름 | 들어 있음 |
| `E` | pycodestyle | PEP 8 스타일(`None` 비교 방식, 줄 길이 등) | 들어 있음 |
| `W` | pycodestyle | PEP 8 경고(잘못된 문자열 이스케이프 등) | 없음 |
| `I` | isort | import 순서 | 들어 있음 |
| `B` | flake8-bugbear | 버그가 나기 쉬운 쓰임 | 없음 |
| `UP` | pyupgrade | 지금 파이썬에서 더 간단히 쓸 수 있는 옛 문법 | 없음 |

어떤 묶음을 검사할지는 `pyproject.toml`의 `select`로 고릅니다. 연습 프로젝트는 `E`, `F`, `I`를 골랐으므로 `W`, `B`, `UP`은 검사하지 않습니다. 쓰고 싶으면 `select`에 글자를 더합니다. `select`를 적지 않으면 Ruff가 정한 기본 규칙(Ruff 0.17.1은 407개)을 검사하는데, 기본 규칙은 Ruff 버전마다 달라지므로 `select`로 적어 두는 편이 안전합니다.

아래는 묶음마다 자주 만나는 규칙입니다. 안 좋은 예의 오른쪽 주석이 Ruff가 짚는 규칙 코드입니다.

**`F`(Pyflakes): 쓰지 않는 것, 없는 이름**

안 좋은 예

```python
import os  # F401: 불러오고 쓰지 않음


def greet(name):
    message = "반가워요"  # F841: 값을 넣고 쓰지 않음
    return f"{nmae}님, 안녕하세요"  # F821: 정의되지 않은 이름(name 오타)
```

고친 예

```python
def greet(name):
    return f"{name}님, 안녕하세요"
```

1-2의 실수 가운데 세 가지입니다. `F401`은 `--fix`가 지워 주고, `F841`은 동작이 바뀔 수 있는 고침이라 `--fix --unsafe-fixes`를 붙여야 지워집니다. `F821`은 어떤 이름을 쓰려 했는지 Ruff가 알 수 없어 직접 고칩니다.

**`E`(pycodestyle): PEP 8 스타일**

안 좋은 예

```python
def has_coupon(coupon):
    if coupon == None:  # E711: None 을 == 로 비교
        return False
    return True
```

고친 예

```python
def has_coupon(coupon):
    if coupon is None:
        return False
    return True
```

`None`은 하나뿐인 값이라 같은 값인지(`==`)가 아니라 바로 그 값인지(`is`)를 봅니다. `==`의 뜻을 바꿔 둔 객체에서는 결과가 달라질 수 있어 `--fix`만으로는 고치지 않고, 직접 고치거나 `--fix --unsafe-fixes`로 고칩니다. 같은 묶음의 `E501`은 한 줄이 `line-length`(연습 프로젝트는 120칸)보다 길 때 나오며, 한글은 한 글자를 2칸으로 셉니다. 어디서 나눌지는 사람이 정할 일이라 직접 나눕니다.

**`W`(pycodestyle 경고): 잘못된 문자열 이스케이프**

안 좋은 예

```python
import re


def find_numbers(text):
    return re.findall("\d+", text)  # W605: 보통 문자열 안의 \d
```

고친 예

```python
import re


def find_numbers(text):
    return re.findall(r"\d+", text)
```

보통 문자열에서 `\`는 `\n`(줄바꿈)처럼 뒤 글자와 묶여 특별한 뜻이 되는데 `\d`는 그런 뜻이 없는 조합이라, 파이썬 3.12부터 경고(`SyntaxWarning`)가 납니다. 앞에 `r`을 붙인 raw 문자열로 쓰면 `\`가 글자 그대로 정규식에 전달됩니다. `--fix`가 고쳐 줍니다.

**`I`(isort): import 순서**

안 좋은 예

```python
from shop.greet import greet  # I001: 프로젝트 모듈을 표준 라이브러리보다 먼저 불러옴
import json


def welcome_json(name):
    return json.dumps({"message": greet(name)})
```

고친 예

```python
import json

from shop.greet import greet


def welcome_json(name):
    return json.dumps({"message": greet(name)})
```

표준 라이브러리, 설치한 외부 라이브러리, 프로젝트 모듈 순서로 묶고 묶음 사이를 빈 줄로 나눕니다. `--fix`가 고쳐 줍니다.

**`B`(flake8-bugbear): 버그가 나기 쉬운 쓰임**

안 좋은 예

```python
def add_item(item, cart=[]):  # B006: 기본값이 리스트
    cart.append(item)
    return cart
```

고친 예

```python
def add_item(item, cart=None):
    if cart is None:
        cart = []
    cart.append(item)
    return cart
```

기본값 `[]`는 함수를 정의할 때 한 번만 만들어져, 장바구니를 넘기지 않은 호출이 모두 그 리스트 하나를 같이 씁니다. 그래서 `add_item("사과")` 다음에 `add_item("배")`를 부르면 `["배"]`가 아니라 `["사과", "배"]`가 나옵니다. 기본값을 `None`으로 두고 함수 안에서 새 리스트를 만듭니다. 동작이 바뀌는 고침이라 `--fix --unsafe-fixes`를 붙여야 Ruff가 이렇게 고쳐 줍니다.

**`UP`(pyupgrade): 옛 문법을 지금 문법으로**

안 좋은 예

```python
def greet(name):
    return "{}님, 안녕하세요".format(name)  # UP032: f-string 으로 쓸 수 있음
```

고친 예

```python
def greet(name):
    return f"{name}님, 안녕하세요"
```

`.format()` 대신 f-string을 쓰면 문자열에 들어갈 값이 그 자리에 바로 보입니다. `--fix`가 고쳐 줍니다.

| 고치는 방법 | 규칙 |
|---|---|
| `--fix`가 바로 고침 | `F401`, `W605`, `I001`, `UP032` |
| `--fix --unsafe-fixes`를 붙여야 고침 | `F841`, `E711`, `B006` |
| 직접 고침 | `F821`, `E501` |

`--fix`는 동작이 바뀌지 않는 고침만 적용합니다. 동작이 바뀔 수 있는 고침은 바뀔 내용을 확인한 뒤 직접 고치거나 `--unsafe-fixes`를 붙입니다(2-2, 2-4).

## 2. Ruff로 검사하고 고치기

### 2-1. 설치 확인과 연습 파일

`uv sync`를 한 뒤 `uv run ruff --version`을 실행하면 `ruff 0.17.1`이 나옵니다.

이 절의 예시는 `pytest_ci_practice` 폴더에 연습 파일 `try_ruff.py`를 만들어 예시 코드를 붙여 넣고, 그 파일 하나만 검사합니다.

- 연습 프로젝트의 코드는 두 검사를 모두 통과하므로, `.`(폴더 전체)로 검사해도 `try_ruff.py`의 결과만 나옵니다.
- 다 따라 한 뒤에는 `try_ruff.py`를 지웁니다. 지우지 않고 올리면 3절의 Ruff 검사 단계가 이 파일 때문에 실패합니다.

### 2-2. 확인하는 명령과 고치는 명령

Ruff 명령은 파일을 그대로 두고 결과만 보여 주는 명령과, 파일을 실제로 고치는 명령으로 나뉩니다.

<img src="../images/ap_21_check_vs_fix.png" width="1050" alt="Ruff 명령을 두 열로 나눈 표. 왼쪽 열 '확인만 한다(파일은 그대로)'에는 린트 ruff check, 포맷 ruff format --check 와 ruff format --diff 가 있고 CI에는 이쪽만 넣습니다. 오른쪽 열 '실제로 고친다(파일이 바뀐다)'에는 린트 ruff check --fix, 포맷 ruff format 이 있고 내 컴퓨터에서만 씁니다.">

| 명령 | 하는 일 | 파일 |
|---|---|---|
| `uv run ruff check .` | 린트 규칙을 어긴 곳을 찾아 보여 줌 | 그대로 |
| `uv run ruff check --fix .` | 그중 안전하게 고칠 수 있는 곳을 고침 | **바뀜** |
| `uv run ruff check --fix --unsafe-fixes .` | 동작이 바뀔 수 있는 고침까지 함 | **바뀜** |
| `uv run ruff format --check .` | 다시 포맷해야 할 파일이 있는지 확인 | 그대로 |
| `uv run ruff format --diff .` | 포맷하면 바뀔 내용을 보여 줌 | 그대로 |
| `uv run ruff format .` | 포맷을 실제로 적용 | **바뀜** |

- **이름이 헷갈리는 곳:** `ruff check`의 check는 린트 검사를 하는 명령 이름이고, `ruff format --check`의 `--check`는 고치지 말고 확인만 하라는 옵션입니다. 둘 다 파일을 바꾸지 않습니다. 파일을 바꾸는 것은 `--fix`를 붙인 `ruff check`와, `--check`나 `--diff` 없이 실행한 `ruff format`입니다.
- **종료 코드:** 확인 명령은 문제가 있으면 종료 코드 1, 없으면 0으로 끝납니다. 부록 01의 pytest처럼 워크플로 단계는 종료 코드가 0이 아니면 실패로 표시되므로, 3절에서는 확인 명령을 워크플로에 넣습니다. 확인 명령은 고치기 전후에 내 컴퓨터에서도 씁니다(2-4).
- **검사 범위:** `.`은 지금 폴더 아래의 파이썬 파일 전체입니다. 파일 하나만 보려면 `.` 대신 파일 이름을 줍니다(이 절의 예시는 `try_ruff.py`).

### 2-3. 검사 결과 읽기

import 순서가 뒤섞이고, 쓰지 않는 `json`을 불러온 코드를 `try_ruff.py`에 붙여 넣습니다.

`try_ruff.py`

```python
from gate_rules import metric_drop
import json


def drop_text(baseline_value, current_value):
    drop = metric_drop(baseline_value, current_value)
    return f"하락폭 {drop}"
```

`uv run ruff check try_ruff.py`를 실행하면 규칙 위반 두 개가 나옵니다.

`uv run ruff check try_ruff.py` 출력

```text
I001 [*] Import block is un-sorted or un-formatted
 --> try_ruff.py:1:1
  |
1 | / from gate_rules import metric_drop
2 | | import json
  | |___________^
help: Organize imports
  |
  - from gate_rules import metric_drop
1 | import json
2 |
3 + from gate_rules import metric_drop
4 +
5 |
  |

F401 [*] `json` imported but unused
 --> try_ruff.py:2:8
  |
1 | from gate_rules import metric_drop
2 | import json
  |        ^^^^
help: Remove unused import: `json`
  |
1 | from gate_rules import metric_drop
  - import json
2 |
  |

Found 2 errors.
[*] 2 fixable with `--fix`.
```

위반 하나가 한 묶음으로 나오고, 묶음은 위에서부터 다음 순서로 읽습니다.

- **첫 줄:** 규칙 코드(`I001`), 자동 고침 표시(`[*]`), 무엇이 문제인지(`Import block is un-sorted or un-formatted`, import 묶음이 정렬되지 않음).
- **`-->` 줄:** 파일 이름, 줄 번호, 칸 번호(`try_ruff.py:2:8`은 2번째 줄 8번째 칸).
- **코드와 `^` 표시:** 문제가 있는 코드 줄과, 그 줄에서 문제인 부분 아래에 `^`를 그어 보여 줍니다.
- **`help:` 아래:** 고치는 방법과, `--fix`를 실행하면 바뀔 내용입니다. `-` 줄은 지워지고 `+` 줄은 새로 생깁니다.
- **마지막 두 줄:** 찾은 위반 수(`Found 2 errors.`)와, 그중 `--fix`로 고칠 수 있는 수(``[*] 2 fixable with `--fix`.``)입니다.

두 위반의 뜻은 이렇습니다.

- `I001`(import 순서): 표준 라이브러리(`json`)를 먼저, 프로젝트 모듈(`gate_rules`)을 나중에 두고 두 묶음 사이를 빈 줄로 나누는 순서가 아닙니다.
- `F401`(쓰지 않는 import): `json`을 불러오고 한 번도 쓰지 않았습니다.

위반이 많으면 `--output-format concise`로 위반 하나를 한 줄로 줄여 볼 수 있습니다.

```text
uv run ruff check --output-format concise try_ruff.py

try_ruff.py:1:1: I001 [*] Import block is un-sorted or un-formatted
try_ruff.py:2:8: F401 [*] `json` imported but unused
Found 2 errors.
[*] 2 fixable with `--fix`.
```

<img src="../images/ap_22_lint_line.png" width="1050" alt="린트 결과 한 줄 try_ruff.py:2:8: F401 [*] `json` imported but unused 를 나눠 읽는 그림. 파일, 줄과 칸, 규칙 코드, 자동 고침 가능 표시, 무엇이 문제인지로 나뉘고, 규칙 코드 F401 은 규칙 묶음 F(Pyflakes)와 묶음 안의 번호 401로 이루어집니다.">

규칙 코드의 뜻이 궁금하면 `uv run ruff rule F401`처럼 규칙 코드를 넣어 실행합니다. 그 규칙이 무엇을 검사하는지, 왜 문제인지, 고친 예시가 영어로 나옵니다.

### 2-4. 고치는 순서

린트 위반과 포맷 차이가 함께 있는 코드를 `try_ruff.py`에 붙여 넣고 차례로 고쳐 봅니다.

```python
from gate_rules import metric_drop
import json
def drop_text(baseline_value,current_value):
    drop=metric_drop(baseline_value,current_value)
    if drop == None:
        return "하락폭 없음"
    return f'하락폭 {drop}'
```

**1단계: 확인한다.** 위반 세 개 가운데 두 개에 `[*]`가 붙었습니다.

```text
uv run ruff check --output-format concise try_ruff.py

try_ruff.py:1:1: I001 [*] Import block is un-sorted or un-formatted
try_ruff.py:2:8: F401 [*] `json` imported but unused
try_ruff.py:5:16: E711 Comparison to `None` should be `cond is None`
Found 3 errors.
[*] 2 fixable with `--fix` (1 unsafe fix available with `--fix --unsafe-fixes`).
```

**2단계: 안전한 고침을 적용한다.** 이 명령은 파일을 실제로 바꿉니다. `I001`, `F401`이 고쳐지고 `E711` 하나가 남습니다.

```text
uv run ruff check --fix --output-format concise try_ruff.py

try_ruff.py:7:16: E711 Comparison to `None` should be `cond is None`
Found 3 errors (2 fixed, 1 remaining).
1 unsafe fix available with `--fix --unsafe-fixes`.
```

`E711`처럼 고침 제안이 있어도 동작이 바뀔 수 있는 것은 **안전하지 않은 고침**(unsafe fix)으로 분류되어 `--fix`만으로는 적용되지 않습니다. pandas처럼 `==`의 동작을 바꿔 둔 라이브러리에서는 `== None`과 `is None`의 결과가 다르기 때문입니다. 예를 들어 pandas의 열에 `== None`을 쓰면 칸마다 비교한 결과가 나오고, `is None`은 열 자체가 `None`인지 봅니다.

**3단계: 남은 위반을 직접 고친다.** `if drop == None:`을 `if drop is None:`으로 바꿉니다. `drop`은 숫자라 `is`로 바꿔도 동작이 같습니다. 바뀔 내용을 확인한 뒤라면 `--fix --unsafe-fixes`로 고쳐도 됩니다.

**4단계: 포맷한다.** 이 명령도 파일을 실제로 바꿉니다. 공백, 따옴표와 2단계의 고침이 맨 위에 남긴 빈 줄이 정리됩니다.

```text
uv run ruff format try_ruff.py

1 file reformatted
```

**5단계: 다시 확인한다.** 두 확인 명령이 모두 통과하면 커밋합니다.

```text
uv run ruff check try_ruff.py

All checks passed!

uv run ruff format --check try_ruff.py

1 file already formatted
```

고친 뒤 `try_ruff.py`입니다.

```python
from gate_rules import metric_drop


def drop_text(baseline_value, current_value):
    drop = metric_drop(baseline_value, current_value)
    if drop is None:
        return "하락폭 없음"
    return f"하락폭 {drop}"
```

- 린트를 먼저 고치고 포맷을 나중에 합니다. `--fix`가 줄을 지우거나 옮기며 남긴 빈 줄과 모양을 포맷이 마지막에 정리하므로, 포맷을 마지막에 하면 한 번에 끝납니다.
- 따라 하기를 마쳤으면 `try_ruff.py`를 지웁니다.

### 2-5. 규칙에서 예외로 두기

린터가 짚은 곳 가운데에는 일부러 그렇게 쓴 코드도 있습니다. 이럴 때는 코드를 억지로 고치지 않고 그 규칙만 예외로 둡니다. 예외를 두는 범위에 따라 세 가지 방법이 있고, 좁은 범위부터 고릅니다.

| 방법 | 범위 | 쓰는 곳 |
|---|---|---|
| `# noqa: 규칙 코드` | 한 줄 | 그 줄 끝 |
| `# ruff: noqa: 규칙 코드` | 파일 하나 전체 | 파일 맨 위 |
| `per-file-ignores` | 이름이 맞는 파일들 | `pyproject.toml` |

예시의 `shop/`은 설명을 위해 만든 작은 패키지이고, 연습 프로젝트에는 없습니다. 예외를 둘 때는 왜 예외인지 바로 옆에 주석으로 남깁니다. 나중에 읽는 사람이 실수로 남은 것인지 일부러 둔 것인지 알 수 있습니다.

**방법 1: 한 줄만 빼기**

패키지 입구 파일(`shop/__init__.py`)에서 안쪽 파일의 함수를 불러 두면, 쓰는 쪽은 `from shop.greet import greet` 대신 `from shop import greet`로 짧게 불러 쓸 수 있습니다. 그런데 이 파일 안에서는 `greet`를 쓰지 않으니 `F401`(쓰지 않는 import)에 걸립니다.

`shop/__init__.py`

```python
from shop.greet import greet
```

```text
uv run ruff check shop/__init__.py

F401 `shop.greet.greet` imported but unused; consider removing, adding to `__all__`, or using a redundant alias
 --> shop/__init__.py:1:24
  |
1 | from shop.greet import greet
  |                        ^^^^^
help: Use an explicit re-export: `greet as greet`

Found 1 error.
```

일부러 둔 import이므로 그 줄 끝에 `# noqa: F401`을 붙입니다. 그 줄의 `F401`만 검사에서 빠지고 `All checks passed!`가 나옵니다.

```python
# 쓰는 쪽에서 from shop import greet 로 바로 불러 쓰게 하려고 불러 둡니다.
from shop.greet import greet  # noqa: F401
```

Ruff의 `help:` 줄처럼 `from shop.greet import greet as greet`로 쓰는 방법도 있습니다. 같은 이름으로 다시 붙이면 "다시 내보내려고 불러 둔 것"이라는 표시로 보고 Ruff가 검사하지 않습니다.

**방법 2: 파일 하나 전체에서 빼기**

화면에 그대로 보여 줄 안내 문장을 모아 둔 파일입니다. 문장이 길어 줄마다 `E501`에 걸립니다.

`shop/messages.py`

```python
WELCOME = "회원 가입을 환영합니다. 첫 주문에는 배송비가 무료이며, 가입 후 7일 안에 사용할 수 있는 10% 할인 쿠폰이 함께 발급됩니다."
REFUND = "환불은 상품을 받은 날로부터 14일 안에 신청할 수 있습니다. 포장을 뜯은 식품과 주문 제작 상품은 환불할 수 없으니 주문 전에 꼭 확인해 주세요."
```

```text
uv run ruff check --output-format concise shop/messages.py

shop/messages.py:1:79: E501 Line too long (131 > 120)
shop/messages.py:2:76: E501 Line too long (149 > 120)
Found 2 errors.
```

문장을 줄마다 나누면 오히려 읽기 어려우므로, 파일 맨 위에 `# ruff: noqa: E501`을 적어 이 파일 전체에서 `E501`만 뺍니다. 다른 규칙은 그대로 검사합니다.

```python
# ruff: noqa: E501
# 화면에 그대로 보여 줄 안내 문장이라 줄을 나누지 않습니다.
WELCOME = "회원 가입을 환영합니다. 첫 주문에는 배송비가 무료이며, 가입 후 7일 안에 사용할 수 있는 10% 할인 쿠폰이 함께 발급됩니다."
REFUND = "환불은 상품을 받은 날로부터 14일 안에 신청할 수 있습니다. 포장을 뜯은 식품과 주문 제작 상품은 환불할 수 없으니 주문 전에 꼭 확인해 주세요."
```

**방법 3: 설정 파일에서 빼기**

같은 예외를 여러 파일에 둬야 하면 파일마다 주석을 다는 대신 `pyproject.toml`의 `[tool.ruff.lint.per-file-ignores]`에 한 번에 적습니다. 왼쪽은 파일 이름이나 경로, 오른쪽은 뺄 규칙 코드 목록입니다. 방법 1·2의 주석 없이도 같은 결과가 나옵니다.

`pyproject.toml`에 추가

```toml
[tool.ruff.lint.per-file-ignores]
"__init__.py" = ["F401"]        # 패키지 입구 파일: 다시 내보내려고 불러만 둠
"shop/messages.py" = ["E501"]   # 화면에 그대로 보여 줄 긴 안내 문장
```

```text
uv run ruff check shop/

All checks passed!
```

- `"__init__.py"`처럼 파일 이름만 쓰면 어느 폴더에 있든 그 이름의 파일에 모두 적용됩니다. 패키지가 여러 개여도 한 줄로 끝납니다.
- 예외가 설정 파일 한 곳에 모여 있어, 프로젝트가 어떤 규칙을 어디서 빼고 있는지 한눈에 봅니다.

**주의: 규칙 코드를 꼭 적습니다**

규칙 코드 없이 `# noqa`만 쓰면 그 줄의 모든 규칙이 빠집니다. 한 줄에 두 모듈을 불러온 코드로 비교합니다. 규칙 코드를 적으면 뺀 규칙(`F401`) 말고 다른 문제는 계속 보입니다.

```python
import os, sys  # noqa: F401
```

```text
uv run ruff check --output-format concise example.py

example.py:1:1: E401 [*] Multiple imports on one line
example.py:1:1: I001 [*] Import block is un-sorted or un-formatted
Found 2 errors.
[*] 2 fixable with `--fix`.
```

`E401`(한 줄에 import 여러 개)과 `I001`(import 순서)이 나왔습니다. 그런데 규칙 코드를 지우고 `# noqa`만 남기면 이 둘까지 사라집니다.

```python
import os, sys  # noqa
```

```text
uv run ruff check example.py

All checks passed!
```

문제가 그대로 있는데 검사는 통과합니다. 예외는 꼭 필요한 규칙만, 규칙 코드를 적어 둡니다.

## 3. 워크플로에 Ruff 검사 넣기

### 3-1. 단계 넣기

부록 02 기초 7절에서 만든 `.github/workflows/ci.yml`의 `unit-tests` 작업 단계 맨 끝(`테스트` 단계 다음)에 아래 Ruff 단계 두 개를 넣습니다. 들여쓰기는 `- name: 테스트`와 같은 앞 여섯 칸입니다.

연습 프로젝트의 `design/ci.yml`에는 이 단계가 이미 들어 있습니다(`[Ruff 추가]` 주석). 기초 7절대로 복사했다면 위치만 확인하고 3-2로 넘어갑니다. 브랜치 만들기부터 올리기까지의 순서는 같은 폴더의 `Ruff_AI리뷰_추가_절차.md`에 있습니다.

`ci.yml`의 `unit-tests` 작업 단계 끝에 들어가는 Ruff 단계

```yaml
      # [Ruff 추가] 코드 스타일 검사 단계. 라이브러리 설치 단계가 dev 묶음의 ruff 까지 설치해 두었음
      - name: 문법·스타일·import 순서 검사
        if: always()                              # 앞의 테스트가 실패해도 이 검사는 실행
        run: uv run ruff check .                  # pyproject.toml 의 select 규칙을 어긴 곳이 있으면 실패
      - name: 포맷 검사
        if: always()
        run: uv run ruff format --check .         # 다시 포맷해야 할 파일이 있으면 실패(파일은 바꾸지 않음)
```

- **확인 명령만 넣습니다.** 러너에서 `--fix`나 `ruff format`으로 고쳐도, 고친 파일은 작업이 끝나면 러너와 함께 사라지고 저장소에는 남지 않습니다. 고치는 명령은 내 컴퓨터에서 실행하고, 고친 파일을 커밋해 올립니다.
- **새 작업을 만들지 않고 단계로 넣었습니다.** 작업마다 새 러너에서 저장소 받기와 설치를 처음부터 다시 하므로, 권한과 실행 조건이 같은 일은 한 작업에 모으면 준비를 한 번만 합니다. `라이브러리 설치` 단계(`uv sync --locked`)가 `dev` 묶음의 Ruff까지 이미 설치합니다.
- **`if: always()`를 붙였습니다.** 한 작업 안에서는 앞 단계가 실패하면 뒤 단계를 건너뜁니다. 테스트가 실패해도 Ruff 검사는 실행되게 해서 한 번의 실행으로 두 문제를 다 봅니다.

### 3-2. 실패와 통과 확인하기

규칙을 어긴 코드를 pull request로 올려 Ruff 검사가 병합을 막는지 확인합니다. 새 브랜치에서 2-3의 코드(import 순서가 뒤섞이고 `json`을 쓰지 않는 코드)를 `try_ruff.py`로 저장해 올립니다.

```bash
git switch main
git pull
git switch -c try/ruff-check
# 2-3 의 코드를 try_ruff.py 로 저장합니다.
git add try_ruff.py
git commit -m "Ruff 검사 확인용 파일 추가"
git push -u origin try/ruff-check
```

main으로 가는 pull request를 열면 다음이 보입니다.

- `unit-tests (ubuntu-latest)`, `unit-tests (windows-latest)` 두 검사가 실패로 표시됩니다. 작업 로그를 열면 `테스트` 단계는 통과하고 `문법·스타일·import 순서 검사` 단계가 실패했으며, 2-3과 같은 `I001`, `F401` 출력이 나옵니다.
- 부록 02 기초 7-3에서 두 검사를 필수 검사로 넣었으므로 병합 버튼이 막힙니다.
- `gate-rules`는 `needs: unit-tests` 때문에 건너뜁니다.
- `ai-review` 작업은 4절에서 다룹니다. 비밀 변수 `OPENAI_API_KEY`를 아직 등록하지 않았다면 이 작업도 실패로 뜨지만, 필수 검사가 아니라 병합에는 영향이 없습니다.

내 컴퓨터에서 2-4의 순서로 고쳐 같은 브랜치에 다시 올립니다.

```bash
uv run ruff check --fix try_ruff.py
uv run ruff format try_ruff.py
git commit -am "try_ruff.py 를 Ruff 규칙에 맞게 고침"
git push
```

같은 pull request에 커밋이 더해져 검사가 다시 돌고, 이번에는 두 `unit-tests` 검사가 통과합니다. `try_ruff.py`는 확인용 파일이므로 병합하지 말고 pull request를 닫습니다(Close pull request).

## 4. AI 코드 리뷰 작업 추가하기

### 4-1. AI 리뷰가 하는 일

pull request를 열면 바뀐 코드를 LLM이 읽고 리뷰를 댓글로 남기게 합니다. 사람이 리뷰하기 전에 놓치기 쉬운 버그나 이름, 빠진 처리 같은 문제를 먼저 짚어 줍니다.

Ruff는 정해진 규칙에 맞는지만 보므로 결과가 늘 같고 통과·실패를 가릅니다. AI 리뷰는 규칙으로 적기 어려운 문제까지 의견으로 말해 주지만, 같은 코드에도 실행마다 다른 말을 할 수 있고 틀린 지적도 섞입니다. 그래서 통과·실패를 가리는 판정이 아니라 참고 의견으로 쓰고, 필수 검사에는 넣지 않습니다.

<img src="../images/ap_23_ai_review_flow.png" width="1050" alt="AI 코드 리뷰가 댓글을 남기는 과정. pull request 열기, git diff 로 바뀐 부분 가져오기, LLM이 읽고 리뷰, review.md 저장, gh pr comment 로 댓글 올리기의 다섯 단계가 이어지고, 판정이 아닌 참고 의견이라 필수 검사에 넣지 않습니다.">

| 단계 | 하는 일 | 파일 |
|---|---|---|
| 바뀐 코드 가져오기 | `git diff origin/main...HEAD`로 이 브랜치에서 바뀐 부분만 가져옴 | `tools/ai_reviewer.py` |
| 리뷰 받기 | 프롬프트와 `ChatOpenAI`로 리뷰를 받아 `review.md`에 저장 | `tools/ai_reviewer.py` |
| 댓글로 올리기 | `gh pr comment`로 `review.md`를 pull request 댓글로 올림 | `ci.yml`의 `ai-review` 작업 |

### 4-2. 리뷰 스크립트

연습 프로젝트의 `tools/ai_reviewer.py`입니다. 워크플로가 실행하는 보조 스크립트라 `tools/` 폴더에 따로 둡니다.

- `origin/main...HEAD`(점 세 개)는 main과 갈라진 지점부터의 변경만 보여 줍니다. 그사이 main에 새로 생긴 커밋은 섞이지 않습니다.
- 바뀐 코드가 없으면 모델을 부르지 않고, 너무 긴 변경은 앞부분 10000자만 보내 비용을 줄입니다.
- API 키는 `.env`에서 읽습니다. `load_dotenv(find_dotenv(usecwd=True))`가 실행한 폴더부터 위로 올라가며 `.env`를 찾아 `OPENAI_API_KEY`를 환경변수로 올립니다. CI에서는 워크플로가 비밀 변수를 환경변수로 넘겨 주고, `load_dotenv`는 이미 있는 환경변수를 덮어쓰지 않으므로 `.env` 없이 그대로 동작합니다.

`tools/ai_reviewer.py`

```python
"""pull request 에서 바뀐 코드(git diff)를 LLM 에게 보여 주고 받은 리뷰를 review.md 로 저장합니다.

GitHub Actions 의 ci 워크플로(ai-review 작업)가 실행하고, review.md 를 pull request 댓글로 올립니다.
로컬 실행(pytest_ci_practice 폴더에서, .env 에 OPENAI_API_KEY 필요):
    git fetch origin main
    uv run python tools/ai_reviewer.py
"""

import subprocess
from pathlib import Path

from dotenv import find_dotenv, load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# 내 컴퓨터에서는 .env 의 OPENAI_API_KEY 를 읽습니다. CI 에서는 워크플로가 비밀 변수를 환경변수로 넘겨 주고,
# load_dotenv 는 이미 있는 환경변수를 덮어쓰지 않으므로 .env 가 없어도 그대로 동작합니다.
load_dotenv(find_dotenv(usecwd=True))

# main 과 갈라진 지점부터 지금까지 이 브랜치에서 바뀐 부분만 가져옵니다(main 에 새로 생긴 커밋은 빠집니다).
diff = subprocess.run(["git", "diff", "origin/main...HEAD"], capture_output=True, text=True, check=True).stdout

if diff.strip() == "":
    review = "main 과 비교해 바뀐 코드가 없어 리뷰하지 않았습니다."
else:
    # 너무 긴 변경은 앞부분 10000자만 보내 모델 입력 길이와 비용을 줄입니다.
    diff = diff[:10000]

    prompt = ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 시니어 파이썬 개발자입니다. 코드 변경(diff)을 읽고 잠재적인 버그와 스타일 문제를 짚고, "
                "잘한 점도 적어 주세요. 한국어 마크다운으로 간결하게 씁니다.",
            ),
            ("human", "다음 코드 변경을 리뷰해 주세요.\n\n```diff\n{diff}\n```"),
        ]
    )
    llm = ChatOpenAI(model="gpt-4.1")
    review = (prompt | llm).invoke({"diff": diff}).content

# 워크플로의 다음 단계가 이 파일을 pull request 댓글로 올립니다.
Path("review.md").write_text(review, encoding="utf-8")
print(review)
```

### 4-3. 내 컴퓨터에서 실행해 보기

워크플로에 넣기 전에 스크립트가 동작하는지 내 컴퓨터에서 먼저 확인합니다. 필요한 라이브러리(`langchain-core`, `langchain-openai`, `python-dotenv`)는 연습 프로젝트의 기본 의존성이라 `uv sync`로 이미 설치되어 있습니다.

1. `pytest_ci_practice` 폴더에 `.env` 파일을 만들고 키를 한 줄 적습니다. `.gitignore`가 `.env`를 막아 저장소에는 올라가지 않습니다.

   ```text
   OPENAI_API_KEY=sk-로 시작하는 키 값
   ```

2. 리뷰할 변경을 만들어 커밋합니다. 스크립트는 main과 비교해 커밋한 변경만 봅니다.

   ```bash
   git switch -c try/ai-review
   # 아무 파일이나 조금 고칩니다(예: citation_check.py 의 주석 한 줄).
   git commit -am "AI 리뷰 확인용 변경"
   ```

3. 비교할 main을 최신으로 받고 스크립트를 실행합니다.

   ```bash
   git fetch origin main
   uv run python tools/ai_reviewer.py
   ```

- 터미널에 리뷰가 출력되고, 같은 내용이 `review.md`로 저장됩니다(`.gitignore`가 막아 올라가지 않음). API를 한 번 부르므로 비용이 듭니다.
- 바뀐 코드가 없으면 모델을 부르지 않고 `main 과 비교해 바뀐 코드가 없어 리뷰하지 않았습니다.`를 출력합니다.
- `.env`가 없거나 키 이름이 다르면 `openai.OpenAIError: Missing credentials`로, 키 값이 틀리면 `AuthenticationError: Error code: 401`로 멈춥니다.
- 다른 폴더의 가상환경이 켜져 있으면 `VIRTUAL_ENV=... will be ignored` 경고가 나옵니다. uv가 이 폴더의 `.venv`를 따로 만들어 쓰므로 무시해도 됩니다.

확인이 끝나면 `git switch main`으로 돌아가고 `git branch -D try/ai-review`로 확인용 브랜치를 지웁니다.

### 4-4. 워크플로에 넣기

1. 부록 02 기초 4-2와 같은 방법으로 저장소 비밀 변수 `OPENAI_API_KEY`를 등록합니다.
2. `ci.yml`의 맨 아래(`gate-rules` 작업 다음)에 아래 `ai-review` 작업을 넣습니다. 들여쓰기는 `gate-rules:`와 같은 앞 두 칸입니다. 연습 프로젝트의 `design/ci.yml`에는 이 작업이 이미 들어 있으므로(`[AI 리뷰 추가]` 주석), 기초 7절대로 복사했다면 위치만 확인합니다.
3. pull request를 열면 `ai-review` 검사가 돌고, 끝나면 pull request에 리뷰 댓글이 달립니다.

작업에서 눈여겨볼 곳입니다.

- `if: github.event_name == 'pull_request'`: 같은 워크플로가 main에 올릴 때와 직접 실행할 때도 돌지만, 리뷰할 pull request가 없으므로 이 작업은 건너뜁니다.
- 별도 작업으로 둔 이유: 이 작업만 pull request 쓰기 권한(`permissions`), 비밀 변수, pull request일 때만 도는 조건이 필요합니다. `unit-tests` 작업과 나누어 권한을 이 작업에만 줍니다.
- `fetch-depth: 0`: 체크아웃은 기본으로 마지막 커밋 하나만 받습니다. 모든 커밋을 받아야 `origin/main`과 비교할 수 있습니다.
- `GH_TOKEN`: 실행마다 GitHub가 자동으로 만들어 주는 토큰(`github.token`)입니다. `OPENAI_API_KEY`와 달리 따로 발급받거나 비밀 변수로 등록할 필요가 없습니다. 러너에 미리 설치된 `gh` 명령이 이 값으로 로그인하고, 권한은 위 `permissions`에 적은 만큼만 받습니다.
- pull request마다 API를 한 번 부르므로 비용이 듭니다. 다른 사람의 포크에서 온 pull request에는 비밀 변수가 전달되지 않아 "AI 리뷰 생성" 단계가 실패합니다.

`ci.yml` 맨 아래에 들어가는 `ai-review` 작업

```yaml
  # [AI 리뷰 추가] pull request 에서 바뀐 코드를 LLM 이 리뷰해 댓글로 남기는 작업. 비밀 변수 OPENAI_API_KEY 필요
  ai-review:
    if: github.event_name == 'pull_request'      # push·직접 실행에서는 건너뜀(리뷰할 pull request 가 없음)
    runs-on: ubuntu-latest
    timeout-minutes: 10
    permissions:
      contents: read
      pull-requests: write                        # 리뷰를 pull request 댓글로 올리려면 필요
    steps:
      - name: 저장소 받기
        uses: actions/checkout@v7
        with:
          fetch-depth: 0                          # 모든 커밋을 받아야 origin/main 과 비교 가능(기본 1은 마지막 커밋만)
      - name: uv 설치
        uses: astral-sh/setup-uv@v10.2.0
        with:
          python-version: "3.12"
      - name: 라이브러리 설치
        run: uv sync --locked                     # uv.lock 에 적힌 버전 그대로 설치(langchain, python-dotenv 포함)
      - name: AI 리뷰 생성
        env:
          OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}   # 부록 02 기초 4-2 처럼 등록한 비밀 변수
        run: uv run python tools/ai_reviewer.py   # review.md 를 만듦
      - name: pull request 댓글로 올리기
        env:
          GH_TOKEN: ${{ github.token }}           # 실행마다 GitHub 가 주는 토큰. gh 명령이 이 값으로 로그인
        run: gh pr comment ${{ github.event.pull_request.number }} --body-file review.md
```